# Example 8.3 / 1: Watch Presolve Reduce the Steel Model

Companion notebook for **8.3 Presolve** in the AMPL lab Documentation, section "Watch Presolve Reduce the Steel Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Watch Presolve Reduce the Steel Model

This version of the steel model adds two one-variable constraints: at least 1000 tons of plate,
and at most 10 hours of coils. `option show_stats 1` prints the size of the problem presolve
sends to the solver. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read
`model.mod`; `ampl.option[...]` sets an AMPL option and `ampl.display(...)` prints the bound
suffixes in AMPL's format:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
subject to Plate_Min: Make['plate'] >= 1000;        # one variable: becomes a bound
subject to Coil_Hours: Make['coils'] / 140 <= 10;   # one variable: becomes a bound

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["show_stats"] = 1
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
ampl.display("Make.lb0", "Make.lb1", "Make.lb2", "Make.lb")
ampl.display("Make.ub0", "Make.ub1", "Make.ub2", "Make.ub")


Presolve eliminates 2 constraints.
Adjusted problem:
3 variables, all linear
1 constraint, all linear; 3 nonzeros
	1 inequality constraint
1 linear objective; 3 nonzeros.

:     Make.lb0 Make.lb1 Make.lb2 Make.lb    :=
bands     0          0        0       0
coils     0          0        0       0
plate     0       1000     1000    1000
;

:     Make.ub0 Make.ub1 Make.ub2 Make.ub    :=
bands    6000     6000     6000    6000
coils    4000     1400     1400    1400
plate    3500     3500     3500    3500
;



### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option show_stats 1;
option solver_msg 0;
option solver highs;
solve;
display Make.lb0, Make.lb1, Make.lb2, Make.lb;
display Make.ub0, Make.ub1, Make.ub2, Make.ub;


Presolve eliminates 2 constraints.
Adjusted problem:
3 variables, all linear
1 constraint, all linear; 3 nonzeros
	1 inequality constraint
1 linear objective; 3 nonzeros.

:     Make.lb0 Make.lb1 Make.lb2 Make.lb    :=
bands     0          0        0       0
coils     0          0        0       0
plate     0       1000     1000    1000
;

:     Make.ub0 Make.ub1 Make.ub2 Make.ub    :=
bands    6000     6000     6000    6000
coils    4000     1400     1400    1400
plate    3500     3500     3500    3500
;



### Another way in amplpy

Presolve turned `Plate_Min` into the lower bound `1000` on `Make['plate']` and `Coil_Hours`
into the upper bound `1400` on `Make['coils']`, then removed both constraints: the solver
received 3 variables and 1 constraint. The suffixes show where each bound came from:

| Suffixes | Meaning | In this run |
|---|---|---|
| `.lb0`, `.ub0` | Bounds declared in the original model, before presolve | `0` and the market sizes |
| `.lb1`, `.ub1` | Bounds inferred from constraints that presolve eliminated | `plate` lower `1000`, `coils` upper `1400` |
| `.lb2`, `.ub2` | The first set plus bounds inferred from constraints retained in the problem | the same here |
| `.lb`, `.ub` | Bounds currently selected for the solver by `presolve` and `var_bounds` | the same here |

With `option presolve 0;` the same model reaches the solver with its 3 constraints. With the
same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.option["show_stats"] = 1
ampl.option["solver_msg"] = 0
ampl.option["presolve"] = 0
ampl.solve(solver="highs")


3 variables, all linear
3 constraints, all linear; 5 nonzeros
	3 inequality constraints
1 linear objective; 3 nonzeros.



### Another way in amplpy

When the data make the problem infeasible, presolve often says so before any solver runs. With
only 5 hours available, `Plate_Min` alone needs 6.25. The first `solve` stops at AMPL's
message; the second one sends the problem to HiGHS:

In [12]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [13]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [14]:
# run.py
ampl.param["avail"] = 5
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print(ampl.solve_result)
ampl.solve(solver="highs")
print(ampl.solve_result)

[WARNING] 
	presolve: constraint Time cannot hold:
		body <= 5 cannot be >= 6.25; difference = -1.25


infeasible

suffix dunbdd OUT;
infeasible
